# Pauli Algebra 03: Hamiltonian Builders

This notebook uses `PauliHamiltonian` as a symbolic builder for spin-1/2 and qubit Hamiltonians.


## What you will learn

1. How `add_spin_term`, `add_pauli_term`, and `add_pauli_op_chain` differ.
2. How spin operators are expanded into Pauli strings.
3. How to inspect term counts and dense matrices for small systems.
4. How to combine prebuilt `PauliString` and `PauliSum` terms with a builder.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_pauli_algebra():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation.pauli_algebra")


try:
    pa = _import_pauli_algebra()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    pa = _import_pauli_algebra()

print("Loaded quantum_simulation.pauli_algebra from", Path(pa.__file__).parent)


Loaded quantum_simulation.pauli_algebra from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation/pauli_algebra


In [2]:
import numpy as np

from quantum_simulation.pauli_algebra import (
    PauliString,
    PauliSum,
    PauliHamiltonian,
    pauli_string_from_sites,
)

np.set_printoptions(precision=4, suppress=True)


## Build a transverse-field Ising chain by hand

`add_spin_term` accepts spin-1/2 operators such as `S_x` and `S_z`. Internally, `S_x = X / 2` and `S_z = Z / 2`.


In [3]:
L = 4
J = 1.0
h = 0.5

tfim = PauliHamiltonian(lattice_length=L)
for i in range(L - 1):
    tfim.add_spin_term(-J, ["S_x", "S_x"], [i, i + 1])
for i in range(L):
    tfim.add_spin_term(-h, ["S_z"], [i])

H_tfim = tfim.build()
print("TFIM term count:", tfim.num_terms())
print(H_tfim)

assert tfim.num_terms() == (L - 1) + L
assert all(len(term.label) == L for term in H_tfim.terms)


TFIM term count: 7
((-0.25+0j))*XXII + ((-0.25+0j))*IXXI + ((-0.25+0j))*IIXX + ((-0.25+0j))*ZIII + ((-0.25+0j))*IZII + ((-0.25+0j))*IIZI + ((-0.25+0j))*IIIZ


## Dense diagnostics for small systems

The builder can still produce a dense matrix through `to_matrix()`. This is intended for small checks, exact diagonalization, or debugging.


In [4]:
H_matrix = tfim.to_matrix()
eigvals = np.linalg.eigvalsh(H_matrix)

print("matrix shape:", H_matrix.shape)
print("Hermitian:", np.allclose(H_matrix, H_matrix.conj().T))
print("lowest eigenvalues:", eigvals[:4])

assert H_matrix.shape == (2**L, 2**L)
assert np.allclose(H_matrix, H_matrix.conj().T)


matrix shape: (16, 16)
Hermitian: True
lowest eigenvalues: [-1.1897 -1.016  -0.6897 -0.516 ]


## Direct Pauli terms and translated chains

Use `add_pauli_term` or `add_pauli_op_chain` when coefficients are already written in terms of Pauli matrices rather than spin operators.


In [5]:
pauli_builder = PauliHamiltonian(lattice_length=4)
pauli_builder.add_pauli_op_chain("XX", coefficient=-1.0, pbc=False)
pauli_builder.add_pauli_term(0.25, ["Z"], [0])
pauli_builder.add_pauli_term(-0.75, ["X", "Z"], [1, 3])

H_pauli = pauli_builder.build()
print("Pauli builder terms:")
for term in H_pauli.terms:
    print(" ", term)

assert pauli_builder.num_terms() == 5
assert any(term.label == "IXIZ" for term in H_pauli.terms)


Pauli builder terms:
  ((-1+0j))*XXII
  ((-1+0j))*IXXI
  ((-1+0j))*IIXX
  ((0.25+0j))*ZIII
  ((-0.75+0j))*IXIZ


## Local spin expansions

Operators such as `S_p`, `S_m`, and `n` expand into multiple Pauli strings. The builder handles the bookkeeping and merges duplicate labels.


In [6]:
spin_builder = PauliHamiltonian(lattice_length=3)
spin_builder.add_spin_term(1.0, ["S_p", "S_m"], [0, 1])
spin_builder.add_spin_term(0.3, ["n"], [2])

H_spin = spin_builder.build()
print("expanded spin terms:")
for term in H_spin.terms:
    print(" ", term)

assert spin_builder.num_terms() == len(H_spin.terms)
assert all(term.label in {"XXI", "XYI", "YXI", "YYI", "III", "IIZ"} for term in H_spin.terms)


expanded spin terms:
  ((0.25+0j))*XXI
  (-0.25j)*XYI
  (0.25j)*YXI
  ((0.25+0j))*YYI
  ((0.15+0j))*III
  ((-0.15+0j))*IIZ


## Mixing existing symbolic terms

`add_term` accepts either a `PauliString` or a `PauliSum`, which makes it convenient to assemble reusable blocks.


In [7]:
block = PauliSum([
    pauli_string_from_sites("ZZ", [0, 2], lattice_length=3, coefficient=0.8),
    PauliString("IXI", -0.2),
])

builder = PauliHamiltonian(lattice_length=3)
builder.add_term(block)
builder.add_pauli_string("III", 1.5)
H = builder.build()

print("combined Hamiltonian:", H)
print("dense trace:", np.trace(H.to_matrix()))

assert builder.num_terms() == 3
assert np.isclose(np.trace(H.to_matrix()), 1.5 * 2**3)


combined Hamiltonian: ((0.8+0j))*ZIZ + ((-0.2+0j))*IXI + ((1.5+0j))*III
dense trace: (12+0j)
